[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssefHosni/prompt-engineering-book-2e/blob/main/notebooks/ch07_task_playbook.ipynb)

# Chapter 7 — The Task Playbook

One input, six tasks. The chapter's argument is that these are not six
techniques; they are one technique with the risk moved around.

On the left of that axis the failure is **omission** — something is dropped
silently and the output still looks complete. On the right it is
**invention**. The middle two should preserve meaning exactly, which makes
them easiest to check and easiest to get quietly wrong.

In [1]:
# Colab only: fetch the shared helpers and install the SDKs.
# Running locally? Skip this cell -- you already have them.
import os, sys
if "google.colab" in sys.modules:
    !pip -q install anthropic openai python-dotenv
    !git clone -q https://github.com/youssefHosni/prompt-engineering-book-2e /content/peb-book
    sys.path.insert(0, "/content/peb-book")
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")


In [2]:
from peb import ask
from peb.fixtures import REVIEW

print(REVIEW)

Ordered the Nordic 400 on the 3rd, arrived on the 11th which is longer than the
3 days advertised. Machine itself is excellent - the burr grinder is much quieter
than my old one and the espresso is genuinely good. Two problems though. The drip
tray warped after about a week, it's very thin plastic. And the manual is clearly
a bad translation, there's a step about "descaling the vapour wand" that makes no
sense. Support answered in about 4 hours which was better than I expected. Would
still recommend it but be prepared to buy a replacement tray. Order ref KL-449021.


In [3]:
def run(label, template, **checks):
    text, _ = ask(template.format(review=REVIEW), max_tokens=4000)
    print(f"--- {label}")
    print(text.strip()[:520])
    for name, fn in checks.items():
        print(f"  [check] {name}: {fn(text)}")
    print()

## §7.1 Summarising: the purpose is the prompt

There is no such thing as *the* summary of a text — only the summary for
somebody, for something. Two readers, same review.

In [4]:
pm = """Summarise this review in 2 sentences for a product manager deciding
what to fix next. Report only faults and their severity. Ignore praise and
logistics.

Review: ```{review}```"""

ops = """Summarise this review in 2 sentences for a logistics lead.
Report only delivery and support performance. Ignore the product itself.

Review: ```{review}```"""

run("summarise (product manager)", pm)
run("summarise (logistics lead)", ops)

--- summarise (product manager)
The drip tray warped within a week due to thin plastic — a hardware durability fault, high severity, as customers expect to buy a replacement part. The manual is a poor translation with at least one incomprehensible maintenance step ("descaling the vapour wand") — a documentation fault, moderate severity, since it blocks correct upkeep of the machine.



--- summarise (logistics lead)
**Delivery and support summary — order ref KL-449021**

Delivery missed the advertised 3-day window: the order was placed on the 3rd and arrived on the 11th, an eight-day transit. Support performance was a positive, with the customer receiving a response in roughly four hours and rating this as better than expected; the only other service-side issue flagged was a poorly translated manual containing an unintelligible descaling instruction.



Those two summaries share no content. Neither is wrong; they answer different
questions, and two clauses made the difference.

## §7.2 Extraction — the keys are fixed, so the check is trivial

In [5]:
extract = """Extract from the review. Use exactly these keys, one per line,
in this order. Use the literal string NONE if a value is absent.

order_ref | ordered_date | arrived_date | support_hours

Review: ```{review}```"""

run("extract", extract,
    all_keys=lambda t: all(k in t for k in
                           ("order_ref", "ordered_date", "arrived_date", "support_hours")))

--- extract
order_ref | KL-449021
ordered_date | the 3rd
arrived_date | the 11th
support_hours | about 4 hours
  [check] all_keys: True



## §7.3 Classification

In [6]:
classify = """Classify this review. Reply with exactly these three lines
and nothing else:

sentiment: positive | negative | mixed
recommends: yes | no
faults: a comma-separated list, or NONE

Review: ```{review}```"""

run("classify", classify,
    three_lines=lambda t: len([l for l in t.strip().splitlines() if l.strip()]) == 3)

--- classify
sentiment: mixed
recommends: yes
faults: late delivery (8 days vs 3 days advertised), drip tray warped after a week, drip tray plastic too thin, poorly translated/confusing manual
  [check] three_lines: True



## §7.4 Transformation — the risk is silent loss

`must_keep` is the whole technique. Decide what must survive, then check it in
code rather than reading the output and feeling reassured.

In [7]:
must_keep = ["KL-449021", "drip tray", "manual", "support"]

transform = """Rewrite the review as an internal support ticket. Neutral register,
no first person, keep every order reference and part name exactly as written.

Review: ```{review}```"""

run("transform", transform,
    kept=lambda t: [k for k in must_keep if k.lower() not in t.lower()] or "all kept")

--- transform
**SUPPORT TICKET**

**Order ref:** KL-449021
**Product:** Nordic 400
**Category:** Delivery delay / product defect / documentation error

---

**Summary**
Customer reports a delivery delay and two post-purchase issues with the Nordic 400. Overall product satisfaction is positive; a recommendation was still given.

**Delivery**
- Order placed on the 3rd; delivery received on the 11th.
- Advertised delivery window was 3 days. Actual transit time exceeded the advertised window.

**Reported issues**
1. **Drip tray** — 
  [check] kept: all kept



## §7.5 Translation — the same check, across a script boundary

In [8]:
translate = """Translate the review into Modern Standard Arabic. Keep product
names, order references and part numbers in Latin script, unchanged.

Review: ```{review}```"""

run("translate", translate,
    latin_kept=lambda t: "KL-449021" in t and "Nordic 400" in t)

--- translate
المراجعة:

طلبت جهاز Nordic 400 في الثالث من الشهر، ووصل في الحادي عشر، أي بمدة أطول من الأيام الثلاثة المُعلَن عنها. الجهاز نفسه ممتاز — فمطحنة الأقراص (burr grinder) أهدأ بكثير من مطحنتي القديمة، والإسبريسو جيد فعلًا. لكن هناك مشكلتان. صينية التنقيط تشوّهت بعد أسبوع تقريبًا، فهي مصنوعة من بلاستيك رقيق جدًا. كما أن دليل الاستخدام ترجمته سيئة بشكل واضح، إذ توجد فيه خطوة عن "إزالة الترسبات الكلسية من عصا البخار" لا معنى لها على الإطلاق. ردّ فريق الدعم في غضون أربع ساعات تقريبًا، وهو أفضل مما توقعت. ما زلت أنصح بشرائ
  [check] latin_kept: True



## §7.6 Expansion — the highest-risk task, so the most constraints

This is the far right of the axis. Every constraint here exists because its
absence produced something wrong once.

In [9]:
expand = """Write a reply to this customer.
Length: 60 to 80 words.
Acknowledge the late delivery and the drip tray.
Offer a replacement tray. Do not offer a refund.
Do not promise a delivery date.
Sign as 'Kestrel Support'.

Review: ```{review}```"""

run("expand", expand,
    length=lambda t: len(t.split()),
    no_refund=lambda t: "refund" not in t.lower(),
    signed=lambda t: "Kestrel Support" in t)

--- expand
Hi,

Thank you for the detailed feedback on order KL-449021. We're sorry the Nordic 400 took longer than the advertised three days to reach you — that fell short of what we aim for. We're also sorry about the warped drip tray, and we'd like to send you a replacement at no cost; just confirm your shipping address and we'll arrange it.

We've passed your note about the manual's descaling section to our documentation team.

Kestrel Support
  [check] length: 77
  [check] no_refund: True
  [check] signed: True



## §7.2 The three-attempt extraction story

The section the chapter calls "the honest case for Chapter 8". Three prompts,
six runs each, scored by a criterion stated in code rather than in prose.

**What `machine-usable` means**, which the chapter asserts and never defines:
all five keys present, and the three numeric fields parseable with `int()`
without cleaning. That is the whole criterion — `ordered_date: 3` passes,
`ordered_date: 3rd` does not, and that single difference is what the three
attempts are about.

All three prompts are printed in full here. The chapter shows one and
paraphrases two, which leaves its conclusion resting on a prompt the reader
cannot see.

In [10]:
import re

from peb import ask
from peb.fixtures import REVIEW

RUNS = 6
KEYS = ["order_ref", "ordered_date", "arrived_date", "support_hours", "faults"]
NUMERIC = ["ordered_date", "arrived_date", "support_hours"]

ATTEMPT_1 = """Extract from the review. Use exactly these keys, one per line,
in this order. Use the literal string NONE if a value is absent.

order_ref | ordered_date | arrived_date | support_hours | faults

Review: ```{review}```"""

ATTEMPT_2 = ATTEMPT_1.replace(
    "Review: ```{review}```",
    # Verbatim from §7.2. An earlier version of this file added ", no ordinals",
    # which is the one hint that fixes `3rd` -- and it took attempt two from 0/6
    # to 6/6, i.e. it silently turned the chapter's three-step story into a
    # two-step one. The prompt under test has to be the prompt the chapter shows.
    "Emit bare values only — no units, no words.\n\n"
    "Review: ```{review}```")

ATTEMPT_3 = """Extract from the review. Emit exactly these five keys, one per
line, in this order, each as `key: value`. Every key must appear even if its
value is NONE.

order_ref       a string, or NONE
ordered_date    an integer day of the month, digits only
arrived_date    an integer day of the month, digits only
support_hours   an integer number of hours, digits only
faults          a comma-separated list, or NONE

Emit bare values only — no units, no words, no ordinals. A value like `3rd` or
`4 hours` is wrong; `3` and `4` are right.

Review: ```{review}```"""


def parse(text):
    """Pull `key: value` pairs out of the reply."""
    out = {}
    for line in text.strip().splitlines():
        m = re.match(r"\s*([a-z_]+)\s*[:|]\s*(.+?)\s*$", line)
        if m and m.group(1) in KEYS:
            out[m.group(1)] = m.group(2).strip()
    return out


def machine_usable(text):
    """All five keys present, and the three numeric fields int()-parseable.

    No cleaning, no regex rescue, no stripping of units. If it needs a repair
    step it is not machine-usable -- that is the point the section is making.
    """
    got = parse(text)
    if set(KEYS) - set(got):
        return False
    for k in NUMERIC:
        try:
            int(got[k])
        except (ValueError, TypeError):
            return False
    return True

In [11]:
print(f"{'attempt':<12} {'machine-usable':>15}   first reply's numeric fields")
for label, prompt in (("one", ATTEMPT_1), ("two", ATTEMPT_2), ("three", ATTEMPT_3)):
    usable = 0
    sample = ""
    for i in range(RUNS):
        text, _ = ask(prompt.format(review=REVIEW), max_tokens=4000)
        usable += machine_usable(text)
        if i == 0:
            got = parse(text)
            sample = "  ".join(f"{k}={got.get(k, '(missing)')!r}" for k in NUMERIC)
    print(f"{label:<12} {usable:>7}/{RUNS:<7}   {sample[:74]}")

print("\nPublished in §7.2: 0 of 6, 0 of 6, 6 of 6.")

attempt       machine-usable   first reply's numeric fields


one                0/6         ordered_date='the 3rd'  arrived_date='the 11th'  support_hours='about 4 ho


two                5/6         ordered_date='3'  arrived_date='11'  support_hours='4'


three              6/6         ordered_date='3'  arrived_date='11'  support_hours='4'

Published in §7.2: 0 of 6, 0 of 6, 6 of 6.


### A note on the field set

The chapter's prompt at §7.2 specifies five keys including `faults`, and its
first result block shows four — `faults` disappears without comment. All three
attempts here keep the five-key set, because dropping the one free-text field
is most of what makes the later attempts tractable, and silently removing it
would be the kind of unstated change the chapter warns about elsewhere.